<a href="https://colab.research.google.com/github/listiantohefiredfsen-debug/fm27-tactics-generator-specialist/blob/main/Qwen_27B_Launcher_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import shutil
import socket
import subprocess
import time
import urllib.request
from google.colab import output

# 1. Cek apakah GPU T4 sudah aktif
if shutil.which("nvidia-smi") is None or subprocess.run(["nvidia-smi"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0:
    raise RuntimeError(
        "GPU belum aktif! Silakan klik menu: Runtime -> Change runtime type -> pilih T4 GPU -> Save, lalu jalankan ulang sel ini."
    )

# 2. Siapkan PATH & install llama.cpp jika belum ada
if "/root/.local/bin" not in os.environ["PATH"]:
    os.environ["PATH"] += ":/root/.local/bin"

llama_bin = shutil.which("llama") or "/root/.local/bin/llama"
if not os.path.exists(llama_bin):
    print("[1/3] Menginstal llama.cpp (tunggu ~20 detik)...")
    subprocess.run("curl -LsSf https://llama.app/install.sh | sh", shell=True, check=True)
    llama_bin = shutil.which("llama") or "/root/.local/bin/llama"
else:
    print("[1/3] llama.cpp sudah siap.")

# 3. Download model hanya jika belum ada di sesi ini
REPO_ID = "DavidAU/Qwen3.8-27B-TURBO-Fable-Cold-Fusion-735-882-Heretic-Uncensored-NEO-CODER-MAX-MTP-GGUF"
FILENAME = "Qwen3.8-27B-TurboFCFusion-735-882-Here-Uncen-NEO-CODER-MAX-MTP-IQ3_M.gguf"
MODEL_DIR = "/content/models"
MODEL_PATH = os.path.join(MODEL_DIR, FILENAME)

if not os.path.exists(MODEL_PATH):
    print("[2/3] Mengunduh model Qwen3.8-27B (sekitar 2–4 menit)...")
    try:
        from huggingface_hub import hf_hub_download
    except ImportError:
        subprocess.run(["pip", "install", "-q", "huggingface_hub"], check=True)
        from huggingface_hub import hf_hub_download

    hf_hub_download(repo_id=REPO_ID, filename=FILENAME, local_dir=MODEL_DIR)
else:
    print("[2/3] File model sudah tersedia, lanjut memuat...")

# 4. Hentikan server lama & cari port kosong
subprocess.run(["pkill", "-f", "llama-server"], stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "-f", "llama server"], stderr=subprocess.DEVNULL)
time.sleep(1)

PORT = None
for p in [8081, 7860, 8000, 8082, 8085]:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        try:
            s.bind(("127.0.0.1", p))
            PORT = p
            break
        except OSError:
            continue

# 5. Jalankan server chat
cmd = [
    llama_bin, "server",
    "-m", MODEL_PATH,
    "-ngl", "99",
    "-c", "4096",
    "--host", "127.0.0.1",
    "--port", str(PORT),
]

log_file = open("/content/server.log", "w")
proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)

print(f"[3/3] Memuat model ke GPU di port {PORT} (tunggu ~30–60 detik)...")
ready = False
for _ in range(120):
    if proc.poll() is not None:
        print("Server berhenti dengan error! Log terakhir:")
        with open("/content/server.log") as f:
            print(f.read()[-1500:])
        break
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        pass
    time.sleep(2)

if ready:
    print("Siap dipakai! Klik link di bawah ini untuk membuka tampilan Chat:")
    output.serve_kernel_port_as_window(PORT)

[1/3] Menginstal llama.cpp (tunggu ~20 detik)...
[2/3] Mengunduh model Qwen3.8-27B (sekitar 2–4 menit)...


Qwen3.8-27B-TurboFCFusion-735-882-Here-U(…): reconstructing file:   0%|          |  0.00B / 14.5GB            

Qwen3.8-27B-TurboFCFusion-735-882-Here-U(…): downloading bytes:           |  0.00B            

[3/3] Memuat model ke GPU di port 8081 (tunggu ~30–60 detik)...
Siap dipakai! Klik link di bawah ini untuk membuka tampilan Chat:
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [2]:
import subprocess
from google.colab import runtime

print("Menghentikan server dan memutus sesi GPU...")
subprocess.run(["pkill", "-f", "llama-server"], stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "-f", "llama server"], stderr=subprocess.DEVNULL)

# Memutus runtime secara total agar kredit langsung berhenti berjalan
runtime.unassign()

Menghentikan server dan memutus sesi GPU...
